# Black-box simulation evaluation

This notebook runs a direct decision / black-box expanding-window evaluation on the simulated macro-financial dataset. The models learn oracle allocation directly and do **not** forecast charge-offs or train PtO/DFL models.

## 1. Imports and setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning

SEED = 42
np.random.seed(SEED)

if "seaborn-v0_8-whitegrid" in plt.style.available:
    plt.style.use("seaborn-v0_8-whitegrid")
elif "seaborn-whitegrid" in plt.style.available:
    plt.style.use("seaborn-whitegrid")
else:
    plt.style.use("default")

pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def find_repo_root(start=None):
    """Find the repository root from repo root, models/simulation, or a notebook subfolder."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "src").exists() and (candidate / "models" / "simulation" / "data").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Set REPO_ROOT manually in this cell.")


REPO_ROOT = find_repo_root()
MODELS_SRC = REPO_ROOT / "models" / "src"
if str(MODELS_SRC) not in sys.path:
    sys.path.insert(0, str(MODELS_SRC))

from capital_allocation_utils import CapitalConfig, optimise_oracle_alpha, realised_utility

print(f"Repo root: {REPO_ROOT}")

## 2. Load simulated data

In [ ]:
simulation_csv = REPO_ROOT / "models" / "simulation" / "data" / "simulated_variables.csv"
df = pd.read_csv(simulation_csv)
df["DATE"] = pd.to_datetime(df["DATE"])
df = df.sort_values("DATE").reset_index(drop=True)

feature_candidates = [
    "sim_persistent_stress_indicator",
    "sim_noisy_leading_indicator",
    "sim_noisy_leading_indicator_lag1",
    "sim_irrelevant_noise",
]
feature_cols = [col for col in feature_candidates if col in df.columns]
if not feature_cols:
    raise ValueError("No simulated feature columns found in the dataset.")

chargeoff_candidates = ["sim_chargeoff", "sim_charge_off_rate", "sim_chargeoff_rate"]
chargeoff_source = next((col for col in chargeoff_candidates if col in df.columns), None)
if chargeoff_source is None:
    raise ValueError(f"Could not find simulated charge-off column. Tried: {chargeoff_candidates}")
df["sim_chargeoff"] = pd.to_numeric(df[chargeoff_source], errors="coerce")

if "regime" in df.columns:
    df["is_stress"] = df["regime"].astype(int)
elif "regime_label" in df.columns:
    df["is_stress"] = df["regime_label"].str.lower().eq("stress").astype(int)
else:
    df["is_stress"] = 0

if "sim_quarterly_pd" in df.columns and "sim_pd_q" not in df.columns:
    df["sim_pd_q"] = df["sim_quarterly_pd"]
if "sim_annual_pd" in df.columns and "sim_pd_a" not in df.columns:
    df["sim_pd_a"] = df["sim_annual_pd"]
if "sim_oracle_alpha" in df.columns:
    df["alpha_oracle_input"] = df["sim_oracle_alpha"]
elif "alpha_oracle" in df.columns:
    df["alpha_oracle_input"] = df["alpha_oracle"]

df = df.dropna(subset=feature_cols + ["sim_chargeoff"]).reset_index(drop=True)

print(f"Loaded {len(df)} quarterly observations from {simulation_csv.relative_to(REPO_ROOT)}")
print(f"Features: {feature_cols}")
print(f"Charge-off source: {chargeoff_source} -> sim_chargeoff")
display(df[["DATE", "is_stress", *feature_cols, "sim_chargeoff"]].head())
display(df[["sim_chargeoff", "is_stress"]].describe().T)

## 3. Define or import decision evaluation functions

The black-box model predicts alpha directly. Realised utility and oracle alpha are evaluated with the same existing capital allocation utilities used elsewhere in the project.

In [ ]:
cap_cfg = CapitalConfig(lgd=0.45)


def clip_alpha(alpha):
    return float(np.clip(alpha, 0.0, 1.0))


def oracle_alpha_for_chargeoff(chargeoff, cfg=cap_cfg):
    return optimise_oracle_alpha(float(chargeoff), cfg)


def regret_from_realisation(alpha_model, alpha_oracle, realised_chargeoff, cfg=cap_cfg):
    oracle_utility = realised_utility(float(alpha_oracle), float(realised_chargeoff), cfg)
    model_utility = realised_utility(float(alpha_model), float(realised_chargeoff), cfg)
    return oracle_utility - model_utility, model_utility, oracle_utility


# Preserve any oracle column loaded from disk as alpha_oracle_input, but use a freshly
# computed oracle alpha for training/evaluation with the current CapitalConfig.
df["alpha_oracle"] = [oracle_alpha_for_chargeoff(c, cap_cfg) for c in df["sim_chargeoff"]]

display(df[["DATE", "sim_chargeoff", "alpha_oracle"]].head())

## 4. Expanding-window evaluation

In [ ]:
initial_train_window = 120
if len(df) <= initial_train_window:
    initial_train_window = max(40, int(np.floor(0.75 * len(df))))

print(f"Initial training window: {initial_train_window} quarters")
print("Training regime counts:")
display(df.iloc[:initial_train_window]["is_stress"].value_counts().rename(index={0: "normal", 1: "stress"}))
print("Test regime counts:")
display(df.iloc[initial_train_window:]["is_stress"].value_counts().rename(index={0: "normal", 1: "stress"}))


def make_models(seed=SEED):
    return {
        "black_box_linear": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "black_box_mlp": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(16, 16),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=2000,
                random_state=seed,
                early_stopping=False,
            )),
        ]),
    }


def run_expanding_window(data, feature_cols, initial_window):
    rows = []

    for t in range(initial_window, len(data)):
        train = data.iloc[:t]
        test = data.iloc[[t]]

        x_train = train[feature_cols]
        y_train = train["alpha_oracle"]
        x_test = test[feature_cols]
        actual_chargeoff = float(test["sim_chargeoff"].iloc[0])
        alpha_oracle = float(test["alpha_oracle"].iloc[0])

        for model_name, model in make_models(seed=SEED + t).items():
            model.fit(x_train, y_train)
            alpha_raw = float(model.predict(x_test)[0])
            alpha_pred = clip_alpha(alpha_raw)
            regret, model_utility, oracle_utility = regret_from_realisation(
                alpha_pred,
                alpha_oracle,
                actual_chargeoff,
                cap_cfg,
            )

            rows.append({
                "DATE": test["DATE"].iloc[0],
                "model": model_name,
                "train_size": t,
                "is_stress": int(test["is_stress"].iloc[0]),
                "regime_label": "stress" if int(test["is_stress"].iloc[0]) == 1 else "normal",
                "actual_chargeoff": actual_chargeoff,
                "actual_pd_q": float(test["sim_pd_q"].iloc[0]) if "sim_pd_q" in test.columns else np.nan,
                "actual_pd_a": float(test["sim_pd_a"].iloc[0]) if "sim_pd_a" in test.columns else np.nan,
                "alpha_oracle": alpha_oracle,
                "alpha_pred_raw": alpha_raw,
                "alpha_pred": alpha_pred,
                "alpha_error": alpha_pred - alpha_oracle,
                "realised_utility": model_utility,
                "oracle_utility": oracle_utility,
                "regret": regret,
            })

    results = pd.DataFrame(rows)
    results["cumulative_regret"] = results.groupby("model")["regret"].cumsum()
    return results


results = run_expanding_window(df, feature_cols, initial_train_window)
display(results.head())
display(results.groupby("model")["DATE"].count().rename("n_test_quarters"))

## 5. Metrics

In [ ]:
def summarise_model(group):
    return pd.Series({
        "alpha_prediction_mse": mean_squared_error(group["alpha_oracle"], group["alpha_pred"]),
        "alpha_prediction_mae": mean_absolute_error(group["alpha_oracle"], group["alpha_pred"]),
        "mean_regret": group["regret"].mean(),
        "median_regret": group["regret"].median(),
        "regret_p95": group["regret"].quantile(0.95),
        "mean_regret_normal": group.loc[group["is_stress"].eq(0), "regret"].mean(),
        "mean_regret_stress": group.loc[group["is_stress"].eq(1), "regret"].mean(),
        "cumulative_regret": group["regret"].sum(),
    })


metrics = results.groupby("model").apply(summarise_model)
display(metrics)

## 6. Plots

In [ ]:
def shade_stress_periods(ax, dates, is_stress, color="tab:red", alpha=0.12):
    dates = pd.Series(pd.to_datetime(dates)).reset_index(drop=True)
    is_stress = pd.Series(is_stress).reset_index(drop=True).astype(bool)
    in_stress = False
    start = None
    for i, stress_now in enumerate(is_stress):
        if stress_now and not in_stress:
            start = dates.iloc[i]
            in_stress = True
        if in_stress and ((not stress_now) or i == len(is_stress) - 1):
            end = dates.iloc[i] if stress_now else dates.iloc[max(i - 1, 0)]
            ax.axvspan(start, end, color=color, alpha=alpha, linewidth=0)
            in_stress = False


test_regimes = df.loc[df["DATE"].isin(results["DATE"].unique()), ["DATE", "is_stress"]]
actual = results.drop_duplicates("DATE")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(actual["DATE"], actual["alpha_oracle"], color="black", label="Oracle alpha")
for model_name, group in results.groupby("model"):
    ax.plot(group["DATE"], group["alpha_pred"], marker="o", markersize=3, label=model_name)
shade_stress_periods(ax, test_regimes["DATE"], test_regimes["is_stress"])
ax.set_title("Predicted alpha vs oracle alpha")
ax.set_ylabel("alpha")
ax.set_xlabel("Date")
ax.set_ylim(-0.02, 1.02)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for model_name, group in results.groupby("model"):
    ax.plot(group["DATE"], group["regret"], marker="o", markersize=3, label=model_name)
shade_stress_periods(ax, test_regimes["DATE"], test_regimes["is_stress"])
ax.set_title("Regret over time")
ax.set_ylabel("Oracle utility - model utility")
ax.set_xlabel("Date")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
for model_name, group in results.groupby("model"):
    ax.plot(group["DATE"], group["cumulative_regret"], marker="o", markersize=3, label=model_name)
shade_stress_periods(ax, test_regimes["DATE"], test_regimes["is_stress"])
ax.set_title("Cumulative regret over time")
ax.set_ylabel("Cumulative regret")
ax.set_xlabel("Date")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for model_name, group in results.groupby("model"):
    ax.scatter(group["alpha_error"], group["regret"], alpha=0.75, label=model_name)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.5)
ax.set_title("Alpha error vs regret")
ax.set_xlabel("Predicted alpha - oracle alpha")
ax.set_ylabel("Regret")
ax.legend()
plt.show()

## 7. Save results

In [ ]:
results_dir = REPO_ROOT / "models" / "simulation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

results_csv = results_dir / "black_box_simulation_results.csv"
results.to_csv(results_csv, index=False)

print(f"Saved black-box simulation results to: {results_csv.relative_to(REPO_ROOT)}")
display(results.head())